# Amazon ML Challenge 2026 — Business Entity Resolution
## 01: Exploratory Data Analysis (EDA)

This notebook inspects the challenge dataset directly and computes real statistics on:
- Record shapes, schema, and column types
- Missing values and duplicate entity IDs
- Country distributions across Source 1, 2, and 3
- Business name and address text distributions
- Ground truth match count distribution (singletons, 1-to-1, 1-to-many matches)

**Note:** No numbers or findings are hard-coded; all figures are computed dynamically.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Setup path to import from src
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    TRAIN_SOURCE1_PATH,
    TRAIN_SOURCE2_PATH,
    TRAIN_SOURCE3_PATH,
    TRAIN_GROUND_TRUTH_PATH,
)
from src.data_loader import load_source_tsv, load_ground_truth

### 1. Load Training Datasets
We load all 3 sources and the ground truth with strict TSV tab-delimitation and string datatypes.

In [ ]:
s1 = load_source_tsv(TRAIN_SOURCE1_PATH, expected_prefix="S1-", source_name="Train Source 1")
s2 = load_source_tsv(TRAIN_SOURCE2_PATH, expected_prefix="S2-", source_name="Train Source 2")
s3 = load_source_tsv(TRAIN_SOURCE3_PATH, expected_prefix="S3-", source_name="Train Source 3")
gt = load_ground_truth(TRAIN_GROUND_TRUTH_PATH)

print("=== Dataset Shapes ===")
print(f"Source 1 (Reference)    : {s1.shape[0]:,} rows, {s1.shape[1]} columns")
print(f"Source 2 (Noisy records): {s2.shape[0]:,} rows, {s2.shape[1]} columns")
print(f"Source 3 (Noisy records): {s3.shape[0]:,} rows, {s3.shape[1]} columns")
print(f"Ground Truth            : {gt.shape[0]:,} rows, {gt.shape[1]} columns")

### 2. Sample Data Inspection

In [ ]:
print("--- Source 1 Sample ---")
display(s1.head(3))

print("--- Source 2 Sample ---")
display(s2.head(3))

print("--- Source 3 Sample ---")
display(s3.head(3))

print("--- Ground Truth Sample ---")
display(gt.head(5))

### 3. Missing Values & Uniqueness Check

In [ ]:
datasets = {"Source 1": s1, "Source 2": s2, "Source 3": s3}

for name, df in datasets.items():
    print(f"\n=== {name} Missing Values & Uniqueness ===")
    print(f"Total Rows: {len(df):,}")
    print(f"Unique Entity IDs: {df['entity_id'].nunique():,}")
    missing = df.isnull().sum() + (df == "").sum()
    pct_missing = (missing / len(df)) * 100
    summary_df = pd.DataFrame({"Missing Count": missing, "Missing %": pct_missing.round(2)})
    display(summary_df)

### 4. Country Distribution
Check all countries present across sources (e.g. US, India, France, etc.) without hardcoding assumptions.

In [ ]:
country_records = []
for name, df in datasets.items():
    vc = df["country"].value_counts(dropna=False)
    for country, count in vc.items():
        country_records.append({"Source": name, "Country": country, "Count": count})

country_df = pd.DataFrame(country_records)
country_pivot = country_df.pivot(index="Country", columns="Source", values="Count").fillna(0).astype(int)
display(country_pivot)

### 5. Text Field Analysis (Name & Address Lengths)

In [ ]:
for name, df in datasets.items():
    name_lens = df["business_name"].astype(str).str.len()
    addr_lens = df["business_address"].astype(str).str.len()
    
    print(f"\n=== {name} Text Statistics ===")
    stats_df = pd.DataFrame({
        "Name Chars": name_lens.describe().round(1),
        "Address Chars": addr_lens.describe().round(1)
    })
    display(stats_df)

### 6. Ground Truth Match Analysis
Inspect the true distribution of matches per Source 1 entity:
- Entities with 0 matches (singletons)
- Entities with 1 match
- Entities with multiple matches
- Breakdown of matches between S2 and S3

In [ ]:
gt["match_count"] = gt["parsed_matched_ids"].apply(len)

print("=== Ground Truth Match Distribution ===")
match_dist = gt["match_count"].value_counts().sort_index()
match_dist_df = pd.DataFrame({
    "Match Count": match_dist.index,
    "S1 Entities": match_dist.values,
    "Percentage (%)": (match_dist.values / len(gt) * 100).round(2)
})
display(match_dist_df)

total_s1 = len(gt)
singletons = int((gt["match_count"] == 0).sum())
single_match = int((gt["match_count"] == 1).sum())
multi_match = int((gt["match_count"] > 1).sum())
max_matches = int(gt["match_count"].max())

print(f"Total Source 1 entities in Ground Truth : {total_s1:,}")
print(f"Singletons (0 matches)                   : {singletons:,} ({singletons/total_s1*100:.2f}%)")
print(f"Single Match (1 match)                   : {single_match:,} ({single_match/total_s1*100:.2f}%)")
print(f"Multi-match (>1 matches)                 : {multi_match:,} ({multi_match/total_s1*100:.2f}%)")
print(f"Maximum matches for a single entity      : {max_matches}")

# Check representation of Source 2 vs Source 3 in matches
all_matched_ids = [m for sublist in gt["parsed_matched_ids"] for m in sublist]
s2_matches = sum(1 for m in all_matched_ids if m.startswith("S2-"))
s3_matches = sum(1 for m in all_matched_ids if m.startswith("S3-"))

print(f"\nTotal Matched Target Entities            : {len(all_matched_ids):,}")
print(f"  - Matched from Source 2                : {s2_matches:,} ({s2_matches/len(all_matched_ids)*100:.2f}%)")
print(f"  - Matched from Source 3                : {s3_matches:,} ({s3_matches/len(all_matched_ids)*100:.2f}%)")